In [1]:
import numpy as np
import os

from google.colab import drive
drive.mount('/content/drive')

os.chdir("/content/drive/MyDrive/Thesis/Models/HMM")

Mounted at /content/drive


In [2]:
a = np.load('HMM_3o_params.npz')
TM = np.log(a['TM']) #transition matrix values (log values), shape 27x27
EM = np.log(a['EM']) #  emission matrix values (log values), shape 27x9261 (3^2 x 21^3)
null_dist = np.log(a['Ends']).reshape(27,1)

 
def HMM_predict(seq):
    seqlen = len(seq)
    
    #initialize
    trellis = np.zeros((seqlen-1, 27), dtype=np.int32)
    mu = null_dist
    
    #step
    for i in range(0,seqlen-2):
        aa_triplet = (seq[i])*(21**2) + (seq[i+1])*21 + seq[i+2]
        update_step = mu + TM + EM[:,aa_triplet]
        mu = update_step.max(axis = 0, keepdims=True).T #need max of every column, in shape (27,1)
        trellis[i-1, : ] = update_step.argmax(axis = 0) #argmax of the column
    
    #last residue
    mu = mu + null_dist
    
    #traceback
    predicted_seq = [None]*seqlen
    backpointer = np.argmax(mu)
    
    predicted_seq[seqlen-1] = backpointer%3
    
    for i in range(seqlen-2,0,-1):
        backpointer = trellis[i,backpointer]
        predicted_seq[i] = backpointer%3
    
    backpointer = trellis[i,backpointer]
    predicted_seq[0] = backpointer//9
        
    return(np.array(predicted_seq))

# def fasta_predict(input_file)
#     outfile = open('HMM_predict.fasta','w')
#     infile  = open('XXX.fasta','r')
    
#     for line in infile:
#         if line[0] == ">":
#             outfile.write(line)
#             HMM_predict(infile.readline().strip())


In [3]:
HMM_preds = {}
# accuracy  = []

# testset = np.load('../../Data/all_test_chains.npz')
testset = np.load('../../Data/n62.npz')
chains = testset.files

for chain in chains:
    HMM_preds[chain] = HMM_predict(testset[chain][0])
    # compare = HMM_preds[chain] == testset[chain][1]
    # accuracy.append (compare.sum()/len(compare))

# print(np.mean(accuracy))
np.savez('../../Data/HMM_predict2.npz', **HMM_preds)